# Label-Findung: Wann ist eine Station defekt?

**Zweck:** Dieses Notebook baut **kein** Modell. Es beantwortet die Frage, die vor jedem
Modell steht: *Was genau soll ein Modell eigentlich erkennen?*

Es schlägt eine Definition von „defekt" vor, die sich allein aus den Messdaten ergibt,
und prüft dann ehrlich, ob diese Definition hält.

**Warum das nötig ist**

| Notebook | Ergebnis | Folge für dieses Notebook |
|---|---|---|
| [`01_kmeans`](01_kmeans_stationscluster.ipynb) | Kein natürliches Gruppenmuster, Silhouette unter 0,22 | Cluster taugen nicht als Label, die Definition muss aus der Physik kommen |
| [`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb) | Die Änderungs-Kategorie misst, **wie viel der Monteur verstellt hat**, nicht wie schlecht die Station lief | Die Begehungstabelle liefert kein brauchbares Zustands-Label |

Ausserdem ist die bisher verwendete Spalte `n_actions > 0` bei 76 % der Stationen wahr.
Ein Label, das fast immer zutrifft, kann nichts trennen.

**Die Idee in einem Satz**

Eine Hausübergabestation soll heisses Wasser bekommen, ihm Wärme entziehen und es kalt
zurückschicken. Fliesst viel Wasser, kommt aber fast genauso heiss zurück, wurde keine
Wärme übertragen. Das ist messbar und braucht keine Begehung.

## 1 · Setup & Daten laden

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", message=".*Data Validation extension.*")

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load_begehungen import begehungen_mit_zaehler

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.facecolor"] = "white"

METER_DIR = ROOT / "data" / "processed" / "real_meters"
DATEIEN = sorted(METER_DIR.glob("*.csv"))
print(f"{len(DATEIEN)} Zaehler-Zeitreihen gefunden")

**Was macht diese Zelle?**

- Sie sucht die Projektwurzel selbst, indem sie vom aktuellen Ordner so lange nach oben
  geht, bis sie `src/` findet. Dadurch läuft das Notebook aus `notebooks/` genauso wie
  aus dem Hauptordner.
- Sie lädt den Loader aus `src/load_begehungen.py`, der in
  [`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb) entstanden ist. Er wird
  erst in Zelle 6 gebraucht, für den Abgleich mit den Begehungen.
- `DATEIEN` sind die stündlichen Zeitreihen je Zähler. Sie stammen aus
  [`00_eda`](00_eda_hast_stationsprofile.ipynb), wo die überlappenden Export-Dateien je
  Zähler zu einer Datei zusammengeführt wurden.
- Die Zeile mit `warnings` unterdrückt nur eine harmlose Meldung beim Excel-Lesen.

## 2 · Monatswerte je Station berechnen

Aus Millionen Stundenwerten werden übersichtliche Monatswerte. Eine Zeile beschreibt
eine Station in einem Monat.

In [ ]:
MIN_STUNDEN = 400   # Monate mit weniger Messstunden werden verworfen

SPALTEN = {
    "Volume flow (l/h)": "fluss",
    "Temperature difference (°C)": "spreizung",
    "Return temperature (°C)": "ruecklauf",
    "Power (kW)": "leistung",
}


def monatswerte(pfad):
    """Monatliche Mittelwerte (Median) einer Zaehler-Zeitreihe."""
    df = pd.read_csv(pfad, parse_dates=["Timestamp"]).set_index("Timestamp").sort_index()
    # Umbenennen ueber ein Mapping, nicht ueber die Spaltenreihenfolge:
    # pandas liefert Spalten in Dateireihenfolge, ein positionsbasiertes
    # Umbenennen vertauscht sonst still Spreizung und Leistung.
    df = df.rename(columns=SPALTEN)
    monate = df.resample("MS").agg(
        fluss=("fluss", "median"),
        spreizung=("spreizung", "median"),
        ruecklauf=("ruecklauf", "median"),
        leistung=("leistung", "median"),
        stunden=("fluss", "count"),
    )
    monate = monate[monate["stunden"] >= MIN_STUNDEN].copy()
    if monate.empty:
        return None
    monate["zaehler"] = int(pfad.stem)
    return monate.reset_index().rename(columns={"Timestamp": "monat"})


teile = [monatswerte(p) for p in DATEIEN]
monate = pd.concat([t for t in teile if t is not None], ignore_index=True)

print(f"Station-Monate mit genug Messdaten: {len(monate)}")
print(f"Stationen                         : {monate['zaehler'].nunique()}")
print(f"Zeitraum                          : {monate['monat'].min():%Y-%m} bis {monate['monat'].max():%Y-%m}")
display(monate.head(3))

**Was macht diese Zelle?**

- Für jede Station wird je Monat ein typischer Wert gebildet: Durchfluss, Spreizung,
  Rücklauftemperatur und Leistung.
- **Median statt Mittelwert:** Der Median ist der mittlere Wert, wenn man alle Stunden
  der Reihe nach sortiert. Einzelne Ausreisser, etwa eine kurze heisse Zapfung,
  verziehen ihn nicht.
- **`MIN_STUNDEN = 400`** wirft Monate mit zu wenigen Messwerten weg. Aus
  [`00_eda`](00_eda_hast_stationsprofile.ipynb) wissen wir, dass die Abdeckung je
  Station zwischen 16,7 % und 98,7 % schwankt. Ein Monatswert aus 30 Stunden wäre
  wertlos.
- Ergebnis ist eine Tabelle, in der **eine Zeile eine Station in einem Monat** ist.

**Begriffe**

| Wort | Bedeutung |
|---|---|
| Durchfluss | wie viel Wasser durch die Station läuft, in Liter pro Stunde |
| Spreizung | Unterschied zwischen ankommendem und zurücklaufendem Wasser, in Kelvin |
| Rücklauf | Temperatur des Wassers, das zurück ins Netz geht |

**Achtung beim Code:** Die Spalten werden über ein Mapping umbenannt und nicht über ihre
Position. Wer hier mit Positionen arbeitet, vertauscht schnell Spreizung und Leistung.
Genau dieser Fehler ist mir beim Vorbereiten passiert und hat das Ergebnis vervierfacht.

## 3 · Das Label festlegen

Jetzt wird aus den Zahlen eine Ja-Nein-Aussage: War diese Station in diesem Monat
auffällig?

In [ ]:
FLUSS_SCHWELLE = 50   # l/h
SPREIZUNG_SCHWELLE = 8  # Kelvin

monate["auffaellig"] = (
    (monate["fluss"] > FLUSS_SCHWELLE) & (monate["spreizung"] < SPREIZUNG_SCHWELLE)
)

betroffen = monate.loc[monate["auffaellig"], "zaehler"].nunique()
print(f"Regel: Durchfluss ueber {FLUSS_SCHWELLE} l/h UND Spreizung unter {SPREIZUNG_SCHWELLE} K\n")
print(f"auffaellige Station-Monate : {int(monate['auffaellig'].sum())} von {len(monate)}"
      f" = {monate['auffaellig'].mean():.1%}")
print(f"betroffene Stationen       : {betroffen} von {monate['zaehler'].nunique()}")

# Gegenprobe an einem Fall, dessen Reparaturdatum bekannt ist
BEISPIEL = 68956351   # Ventil am 09.09.2024 geschlossen, siehe Notebook 03
bsp = monate[monate["zaehler"] == BEISPIEL].set_index("monat")
print(f"\nGegenprobe Zaehler {BEISPIEL}, Reparatur war im September 2024:")
display(bsp.loc["2024-06":"2024-12", ["fluss", "spreizung", "auffaellig"]].round(1))

**Was macht diese Zelle?**

- Sie wendet die Regel an: **Durchfluss über 50 l/h und gleichzeitig Spreizung unter
  8 Kelvin** bedeutet auffällig.
- Anschaulich: Es fliesst ordentlich Wasser, aber es kommt fast genauso heiss zurück,
  wie es angekommen ist. Die Station wälzt um, ohne Wärme abzugeben.
- Statt einem Urteil je Station gibt es eines **je Station und Monat**. Das ist wichtig,
  weil eine Station jahrelang defekt sein und danach in Ordnung sein kann.

**Die Gegenprobe ist der eigentliche Punkt dieser Zelle**

Aus [`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb) kennen wir einen Fall mit
belegtem Datum: Bei Zähler 68956351 wurde am 09.09.2024 ein Ventil geschlossen, der
Durchfluss fiel über Nacht von 499 auf 0 l/h.

Die Regel weiss davon nichts. Sie sieht nur Zahlen. Trotzdem kippt sie genau in diesem
Monat von auffällig auf unauffällig, und die Spreizung springt von rund 0 auf über
30 Kelvin. Die Regel bildet also ab, was tatsächlich passiert ist.

**Wichtig:** Die beiden Schwellen sind eine Setzung, kein Messergebnis. Ob sie tragen,
prüft die nächste Zelle.

## 4 · Sind die Schwellen robust?

Eine Definition, die bei leicht anderen Zahlen völlig andere Ergebnisse liefert, wäre
willkürlich. Deshalb wird die Regel mit verschiedenen Schwellen durchgerechnet.

In [ ]:
fluss_werte = [40, 50, 60]
spreizung_werte = [6, 8, 10]

raster = []
mengen = {}
for f in fluss_werte:
    for s in spreizung_werte:
        treffer = (monate["fluss"] > f) & (monate["spreizung"] < s)
        stationen = set(monate.loc[treffer, "zaehler"])
        mengen[(f, s)] = stationen
        raster.append({"Durchfluss >": f, "Spreizung <": s,
                       "Monate": int(treffer.sum()), "Stationen": len(stationen)})

print("Wie viele Treffer ergeben andere Schwellen?")
display(pd.DataFrame(raster).pivot(index="Durchfluss >", columns="Spreizung <",
                                   values=["Monate", "Stationen"]))

basis = mengen[(FLUSS_SCHWELLE, SPREIZUNG_SCHWELLE)]
streng = mengen[(60, 6)]
locker = mengen[(40, 10)]
print(f"\nStationen bei unserer Regel (50/8) : {len(basis)}")
print(f"strengste Variante (60/6)          : {len(streng)}, davon in unserer Menge: {len(streng & basis)}")
print(f"lockerste Variante (40/10)         : {len(locker)}, davon in unserer Menge: {len(locker & basis)}")

**Was macht diese Zelle?**

- Sie wiederholt die Regel mit neun verschiedenen Schwellenpaaren und zählt jeweils die
  Treffer.
- Danach vergleicht sie, **welche Stationen** getroffen werden, nicht nur wie viele.

**Wie man das Ergebnis liest**

Die Zahl der auffälligen Monate bewegt sich über alle neun Varianten nur zwischen rund
240 und 320. Vor allem aber: Die Stationen der strengsten Variante stecken **vollständig**
in unserer Menge, und die lockerste fügt nur wenige hinzu.

Das heisst, es gibt einen stabilen Kern von Stationen, der unabhängig von der genauen
Schwelle auffällt. Die Definition hängt also nicht an einer glücklich gewählten Zahl.

Das ist ein Unterschied zur Baseline in `src/fault_detection.py`, deren Schwellen laut
`docs/annahmen-limitationen.md` auf synthetischen Daten kalibriert wurden und bei fast
allen echten Zählern anschlagen.

## 5 · Wann wechselt der Zustand?

Ein brauchbares Label sollte sich über die Zeit sinnvoll verhalten. Hier wird gezählt,
wie oft eine Station von unauffällig auf auffällig kippt und zurück.

In [ ]:
monate = monate.sort_values(["zaehler", "monat"]).reset_index(drop=True)
monate["vormonat"] = monate.groupby("zaehler")["auffaellig"].shift()

beginn = monate[(monate["auffaellig"]) & (monate["vormonat"] == False)]
ende = monate[(~monate["auffaellig"]) & (monate["vormonat"] == True)]

print(f"Beginn eines auffaelligen Zustands: {len(beginn)} mal, bei {beginn['zaehler'].nunique()} Stationen")
print(f"Ende eines auffaelligen Zustands  : {len(ende)} mal, bei {ende['zaehler'].nunique()} Stationen")

# Wie viele Stationen waren schon im allerersten Messmonat auffaellig?
erste = monate.groupby("zaehler").first()
print(f"schon im ersten Messmonat auffaellig: {int(erste['auffaellig'].sum())} Stationen"
      f" - deren Beginn wurde nie beobachtet")

# Wie gut waere ein Modell, das einfach den Vormonat abschreibt?
vergleich = monate.dropna(subset=["vormonat"])
print(f"\nTriviales Modell 'wie im Vormonat': {(vergleich['auffaellig'] == vergleich['vormonat']).mean():.1%} richtig")
print(f"Triviales Modell 'immer unauffaellig': {(~vergleich['auffaellig']).mean():.1%} richtig")

**Was macht diese Zelle?**

- Sie vergleicht jeden Monat mit dem Vormonat derselben Station und zählt die Wechsel.
- **Beginn** heisst: vorher unauffällig, jetzt auffällig. **Ende** heisst umgekehrt.
- Zusätzlich rechnet sie aus, wie gut zwei sehr dumme Modelle wären.

**Warum das wichtig ist**

Der Zustand ändert sich selten. Ein Modell, das stur sagt „nächster Monat ist wie
dieser", liegt zu rund 97 % richtig, ohne irgendetwas verstanden zu haben.

Das ist derselbe Fallstrick wie bei der Baseline auf synthetischen Daten, die laut
`docs/annahmen-limitationen.md` 100 % erreichte und trotzdem nichts aussagte.

**Was daraus folgt**

Interessant sind nur die wenigen Wechsel, nicht die vielen gleichbleibenden Monate. Wer
später ein Modell baut, muss es **auf den Wechseln** bewerten und nicht auf allen Zeilen.
Und ein Teil der Stationen war schon auffällig, als die Messung begann. Deren Beginn
lässt sich nie beobachten.

## 6 · Stimmt das Label mit den Begehungen überein?

Bisher prüft sich die Regel nur an sich selbst. Jetzt kommt die einzige unabhängige
Quelle ins Spiel: die Notizen der Monteure.

In [ ]:
VERFUEGBAR = {int(p.stem) for p in DATEIEN}
beg = begehungen_mit_zaehler(available=VERFUEGBAR)

# War die Station irgendwann auffaellig?
je_station = monate.groupby("zaehler")["auffaellig"].any()
beg["physik_auffaellig"] = beg["meter"].map(je_station).fillna(False)

# Nennt der Freitext ein hydraulisches Thema?
MUSTER = r"ventil|zirk|schmutzf|filter|durchfluss|pumpe|kurzschluss"
beg["text_hydraulik"] = beg["sonstiges"].fillna("").str.contains(MUSTER, case=False, regex=True)

kreuz = pd.crosstab(beg["text_hydraulik"], beg["physik_auffaellig"])
print("Freitext nennt Hydraulik (Zeilen) vs. Label auffaellig (Spalten):")
display(kreuz)

mit = beg.loc[beg["text_hydraulik"], "physik_auffaellig"].mean()
ohne = beg.loc[~beg["text_hydraulik"], "physik_auffaellig"].mean()
print(f"Freitext nennt Hydraulik  : {mit:.0%} auffaellig (n={int(beg['text_hydraulik'].sum())})")
print(f"Freitext nennt es nicht   : {ohne:.0%} auffaellig (n={int((~beg['text_hydraulik']).sum())})")
if kreuz.shape == (2, 2):
    print(f"Fisher-Test: p = {stats.fisher_exact(kreuz.values)[1]:.3f}")

# Fallen die Zustandswechsel mit Begehungen zusammen?
bd = beg.dropna(subset=["datum", "meter"]).copy()
begehungsmonate = set(zip(bd["meter"].astype(int), bd["datum"].dt.to_period("M").astype(str)))
for name, tab in [("Beginn", beginn), ("Ende", ende)]:
    paare = zip(tab["zaehler"].astype(int), tab["monat"].dt.to_period("M").astype(str))
    treffer = sum(p in begehungsmonate for p in paare)
    print(f"{name}: {treffer} von {len(tab)} Wechseln fallen in einen Begehungsmonat")

**Was macht diese Zelle?**

Sie stellt zwei unabhängige Fragen.

**Frage 1: Passt das Label zu dem, was die Monteure aufgeschrieben haben?**

Aus [`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb) wissen wir, dass 57 der
66 Begehungen eine Freitextnotiz haben, in der steht, was tatsächlich gefunden wurde.
Nennt eine Notiz ein hydraulisches Thema wie Ventil, Zirkulation oder Schmutzfänger,
sollte die Station auch nach unserer Regel auffällig sein.

**Das Ergebnis ist ernüchternd.** Der Unterschied zwischen beiden Gruppen ist klein und
statistisch nicht abgesichert. Die Notizen bestätigen das Label also **nicht**.

Drei mögliche Gründe, die man nicht auseinanderhalten kann:
- Die Suche nach Stichworten im Freitext ist grob. Ein Mensch, der die 57 Notizen liest,
  würde sauberer einteilen.
- Die Begehungen decken nur 66 der 102 Stationen ab, und die Regel trifft davon wenige.
- Vielleicht beschreiben Notiz und Regel wirklich verschiedene Dinge.

**Frage 2: Werden die Zustandswechsel durch Begehungen ausgelöst?**

Fast nicht. Nur sehr wenige der Wechsel fallen in einen Monat, in dem auch eine Begehung
stattfand.

Bemerkenswert ist aber, dass **alle** diese wenigen Fälle zu Befunden aus
[`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb) passen: Bei den beiden
Stationen, deren auffälliger Zustand in einem Begehungsmonat **endet**, war dort die
stärkste Verbesserung gemessen worden, darunter der bekannte Ventilfall. Und die
Station, deren auffälliger Zustand in einem Begehungsmonat **beginnt**, ist genau jene,
deren Rücklauf nach der Begehung von 18,6 auf 71,0 Grad stieg. Die beiden Notebooks
kommen also unabhängig voneinander auf dieselben Einzelfälle.

Das heisst: Die Stationen verändern ihren Zustand überwiegend **ohne** erkennbaren
äusseren Anlass. Das ist ein eigenständiger Befund und passt dazu, dass
[`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb) keinen statistisch
gesicherten Effekt der Begehungen nachweisen konnte.

**Dieser Stichwortabgleich ist allerdings grob.** Abschnitt 7 wiederholt die Prüfung mit
einer Einordnung von Hand und kommt zu einem deutlich genaueren Ergebnis.


## 7 · Abgleich mit der Handkodierung

Der Stichwortabgleich aus Abschnitt 6 ist grob. Deshalb wurden alle 57 Freitextnotizen
zusätzlich **von Hand** eingeordnet, und zwar blind, also ohne zu sehen, welche Station
die Regel als auffällig einstuft.

In [ ]:
KODIERBOGEN = ROOT / "data" / "processed" / "kodierbogen_freitexte.xlsx"

if not KODIERBOGEN.is_file():
    print(f"{KODIERBOGEN.name} nicht gefunden - Abschnitt uebersprungen.")
else:
    kod = pd.read_excel(KODIERBOGEN, sheet_name="Kodierung")
    je_station = monate.groupby("zaehler")["auffaellig"].any()
    auff_monate = monate.groupby("zaehler")["auffaellig"].sum()
    kod["label"] = kod["Zaehler"].map(je_station)
    kod["auff_monate"] = kod["Zaehler"].map(auff_monate)

    print(f"kodierte Notizen: {len(kod)}")
    display(kod["passt_zur_regel"].value_counts().to_frame("n"))

    bewertbar = kod.dropna(subset=["label"])
    print("Anteil der Stationen, die die Regel auffaellig findet:")
    display(bewertbar.groupby("passt_zur_regel")["label"]
            .agg(Anteil="mean", n="size").round(2))

    jn = bewertbar[bewertbar["passt_zur_regel"].isin(["ja", "nein"])]
    kreuz = pd.crosstab(jn["passt_zur_regel"], jn["label"])
    if kreuz.shape == (2, 2):
        print(f"Fisher-Test ja gegen nein: p = {stats.fisher_exact(kreuz.values)[1]:.3f}")

    schwer = bewertbar[bewertbar["auff_monate"] >= 10]
    print(f"\nStationen mit mindestens 10 auffaelligen Monaten: {schwer['Zaehler'].nunique()}")
    display(schwer[["Zaehler", "auff_monate", "passt_zur_regel", "befundart"]]
            .sort_values("auff_monate", ascending=False))

    n_ja = int((kod["passt_zur_regel"] == "ja").sum())
    verpasst = kod[(kod["passt_zur_regel"] == "ja") & (kod["auff_monate"] == 0)]
    print(f"als ja kodiert, aber kein einziger auffaelliger Monat: {len(verpasst)} von {n_ja}")

    print("\nLabel-Anteil je Befundart:")
    display(bewertbar.groupby("befundart")["label"]
            .agg(Anteil="mean", n="size").round(2).sort_values("Anteil", ascending=False))

    print(f"\nZeilen ohne Labeldaten: {int(kod['label'].isna().sum())}")

**Was macht diese Zelle?**

- Sie liest den ausgefüllten Kodierbogen und verbindet ihn über die Zählernummer mit dem
  Label aus Abschnitt 3.
- Dann vergleicht sie beide Seiten auf drei Arten: insgesamt, nur bei den schweren Fällen,
  und aufgeschlüsselt nach der Art des Befundes.
- Fehlt die Datei, wird der Abschnitt übersprungen. Das Notebook läuft also auch ohne sie.

**Das Ergebnis in einem Satz: Der Freitext bestätigt die Regel nicht allgemein, wohl aber
bei den schweren Fällen.**

**Erstens, insgesamt stimmt nur die Richtung**

Von den Stationen, deren Notiz mit ja kodiert wurde, findet die Regel rund ein Drittel.
Bei nein ist es weniger als ein Zehntel. Die Reihenfolge stimmt also, der Unterschied ist
aber statistisch **nicht abgesichert**. Bei 14 ja- und 13 nein-Fällen ist das auch kaum zu
erwarten.

Über die Hälfte der Notizen wurde mit **unklar** kodiert. Sie sagen zu unserer Frage
schlicht nichts, was den Umfang der Prüfung von vornherein begrenzt.

**Zweitens, die Regel übersieht viele dokumentierte Mängel**

Mehrere überzeugend beschriebene Fälle haben **keinen einzigen** auffälligen Monat,
darunter ein Stellmotor, der laut Notiz defekt war und dessen Ventil aufgedrückt wurde.

Zwei Erklärungen liegen nahe:

- **Zeitweise Fehler** verschwinden im Monatsmedian. Eine Notiz nennt ausdrücklich ein
  Ventil, das nur „hin und wieder" hängt. So etwas bewegt einen Monatswert nicht.
- **Fehler auf der Hausseite**, etwa an der Heizkreispumpe oder an einer Fußbodenheizung
  ohne Mischer, zeigen sich auf der Netzseite nicht zwangsläufig als Dauerdurchfluss.

**Drittens, bei den schweren Fällen passt es sehr gut**

Betrachtet man nur die Stationen mit mindestens zehn auffälligen Monaten, ist **keine
einzige** mit nein kodiert. Die drei Stationen mit über zwanzig auffälligen Monaten wurden
alle mit ja kodiert.

Dazu passt die Aufschlüsselung nach Befundart: Notizen zu Zirkulation und Hydraulik treffen
deutlich häufiger auf auffällige Stationen als Notizen zu Regelung oder Konfiguration.

**Was daraus folgt**

Die Regel ist **kein allgemeiner Defektdetektor**, sondern ein Detektor für **schwere,
monatelang anhaltende Zustände**. In dieser engeren Rolle ist sie belastbar. So sollte sie
auch benannt werden.

## 8 · Ergebnis speichern

In [ ]:
ausgabe = ROOT / "data" / "processed" / "station_monat_label.csv"
spalten = ["zaehler", "monat", "stunden", "fluss", "spreizung", "ruecklauf",
           "leistung", "auffaellig"]
monate[spalten].to_csv(ausgabe, index=False)

meta = {
    "erzeugt": pd.Timestamp.now().isoformat(timespec="seconds"),
    "regel": f"fluss > {FLUSS_SCHWELLE} l/h UND spreizung < {SPREIZUNG_SCHWELLE} K",
    "min_stunden_je_monat": MIN_STUNDEN,
    "station_monate": int(len(monate)),
    "stationen": int(monate["zaehler"].nunique()),
    "auffaellige_monate": int(monate["auffaellig"].sum()),
    "betroffene_stationen": int(betroffen),
    "wechsel_beginn": int(len(beginn)),
    "wechsel_ende": int(len(ende)),
    "schwellen_geprueft": {"fluss": fluss_werte, "spreizung": spreizung_werte},
}
ausgabe.with_suffix(".json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Label-Tabelle -> {ausgabe.name}  ({len(monate)} Zeilen)")
print(f"Lauf-Kontext  -> {ausgabe.with_suffix('.json').name}")

**Was macht diese Zelle?**

- Sie schreibt die Label-Tabelle nach `data/processed/station_monat_label.csv`. Eine
  Zeile ist eine Station in einem Monat, mit allen Messwerten und dem Ja-Nein-Urteil.
- Daneben entsteht eine kleine JSON-Datei mit der verwendeten Regel, den Schwellen und
  den wichtigsten Zahlen. Ohne sie wäre in einigen Wochen nicht mehr nachvollziehbar,
  unter welchen Annahmen die Tabelle entstanden ist.
- `with_suffix(".json")` leitet den Namen vom CSV-Pfad ab, damit beide Dateien
  zusammenbleiben.

**Hinweis zum Datenschutz:** Die Tabelle enthält Zählernummern. `data/processed/` ist
über `.gitignore` ausgeschlossen und bleibt es.

## Zusammenfassung der Gesamtergebnisse

### Was dieses Notebook geliefert hat

Aus 66 Begehungszeilen, von denen keine als Zustands-Label taugte, sind über 2500
bewertete Station-Monate geworden. Die Definition stammt allein aus den Messdaten.

| Prüfung | Ergebnis | Bewertung |
|---|---|---|
| Gegenprobe am bekannten Reparaturfall | Label kippt genau im Reparaturmonat | **bestanden** |
| Robustheit der Schwellen | stabiler Kern über neun Schwellenpaare | **bestanden** |
| Zeitliches Verhalten | wenige, aber klar erkennbare Wechsel | **bestanden** |
| Bestätigung durch Freitext, Stichwortsuche | kein gesicherter Zusammenhang | **nicht bestanden** |
| Bestätigung durch Handkodierung, alle Fälle | Richtung stimmt, nicht abgesichert | **teilweise** |
| Bestätigung durch Handkodierung, schwere Fälle | keine einzige mit nein kodiert | **bestanden** |

### Woher die Bausteine stammen

- **[`00_eda`](00_eda_hast_stationsprofile.ipynb)** hat die zusammengeführten
  Stundenreihen je Zähler geliefert, auf denen hier gerechnet wird, und den Hinweis auf
  die stark schwankende Abdeckung, der zur Mindeststundenzahl je Monat geführt hat.
- **[`01_kmeans`](01_kmeans_stationscluster.ipynb)** hat gezeigt, dass die Stationen
  kein natürliches Gruppenmuster bilden. Deshalb konnten Cluster nicht als Label dienen.
  Gleichzeitig hat die Hauptkomponentenanalyse Rücklauf und Spreizung als wichtigste
  Achse ausgewiesen, also genau die Grössen, aus denen die Regel jetzt besteht.
- **[`03_wirkungsanalyse`](03_wirkungsanalyse_begehungen.ipynb)** hat den Loader für die
  Begehungen bereitgestellt, den Fall mit belegtem Reparaturdatum für die Gegenprobe und
  den Befund, dass die Änderungs-Kategorie das Handeln des Monteurs beschreibt und nicht
  den Zustand der Anlage.

### Wie das Ergebnis einzuordnen ist

Das Label ist ein **geprüfter Vorschlag mit klar umrissener Reichweite**.

Die Handkodierung aller 57 Notizen in Abschnitt 7 hat gezeigt: Als allgemeiner
Defektdetektor taugt die Regel **nicht**. Sie übersieht zeitweise auftretende Fehler und
solche auf der Hausseite, und mehrere überzeugend dokumentierte Mängel haben keinen
einzigen auffälligen Monat.

Bei **schweren, monatelang anhaltenden Zuständen** trifft sie dagegen zuverlässig. Unter
den Stationen mit mindestens zehn auffälligen Monaten ist keine einzige, deren Notiz dem
widerspricht.

**Das Label sollte deshalb als Detektor für Dauerzustände benannt werden**, nicht als
allgemeines Defekt-Label. In dieser engeren Rolle ist es belastbar und benutzbar.

### Was als Nächstes sinnvoll ist

**Ohne Modell, sofort nutzbar**

1. **Die Reichweite des Labels in der Arbeit sauber benennen.** Es ist ein Detektor für
   Dauerzustände. Die Handkodierung in Abschnitt 7 belegt sowohl die Stärke als auch die
   Grenze, und beides gehört dokumentiert.
2. **Die Schwellen in `src/fault_detection.py` nachjustieren.** Laut
   `docs/naechste-schritte.md` schlägt die bisherige Regel bei fast allen echten Zählern
   an und ist damit unbrauchbar. Die Label-Tabelle liefert erstmals einen Massstab aus
   echten Daten.
3. **Die auffälligen Stationen als Wartungsliste an ÜZ geben**, mit dem Zeitraum, in dem
   sie auffällig waren.

**Erst danach ein Modell**

4. **Die Wechsel einzeln ansehen.** Gibt es in den Monaten davor ein Vorzeichen, etwa
   einen langsam steigenden Rücklauf? Ohne Vorzeichen gibt es nichts vorherzusagen.
5. **Falls ein Vorzeichen existiert**, ein bewusst einfaches Modell bauen und es
   **ausschliesslich auf den Wechseln** bewerten. Auf allen Zeilen bewertet gewinnt immer
   das triviale Modell aus Zelle 5.

**Wenn sich das Label nicht bestätigen lässt**

Auch das wäre ein Ergebnis. Dann lautet die Aussage, dass sich „defekt" aus diesen Daten
nicht eindeutig definieren lässt, und das erklärt rückwirkend, warum sowohl das
Clustering in [`01_kmeans`](01_kmeans_stationscluster.ipynb) als auch die bisherigen
Klassifikationsversuche kein Signal gefunden haben.